# Stage 01 — Review (T1 + T3 + M1/M2/M3)

For every patent in `matched/` (output of Stage 00b2 — figures already cropped
**and already matched to BRIEF DESCRIPTION lines**, including
`matched_description`/`match_status`/`fig_number` written into
`crops_mapping.csv`):
1. Read each crop's precomputed match result from `crops_mapping.csv` (no
   re-matching here — see 00b2's matching cell, which has the filenames and
   `data/descriptions.csv` available right after cropping)
2. SigLIP zero-shot: T2 (per-figure visual fields), G1 (architecture topType),
   M1 (fuselage/gear/symmetry), M2 (wing config/empennage), M3 (propulsion)
3. PatentSBERTa: T1 dimensions (scope, field, target) — uses the patent's full
   description text from `data/descriptions.csv` directly (a separate use of
   that file from the per-figure matching in step 1, which now lives in 00b2)
4. Export the consolidated `source_patents.xlsx` (every patent's record is
   returned in-memory only — no per-patent JSON/HTML files are written here)

Human review happens in the single master wizard
(`UI_for_taxonomy_caracterization_10.0.html`): load a patent's
`ai_prelabel.json` (from the AI pre-labeling cell below) via its "AI Pre-Label"
file input, confirm/correct fields, then export. There is no per-patent
review-HTML generation step in this notebook.

SigLIP and PatentSBERTa are **required** — the notebook will not run without them.

**match_status colour key** (computed in 00b2, read here as-is):
- 🟢 `matched` — filename label found and matched to a description line (confidence 0.95)
- 🔵 `semantic` — filename label not found verbatim; SBERT picked the closest description line
- 🔴 `unmatched` — filename label found but no matching line in description text (confidence 0.0)
- 🟡 `no_label` — no label in filename; positional fallback used (confidence 0.3)
- 🟠 `duplicate` — same figure number claimed by multiple images (confidence 0.9)
- ⚫ `human_required` — no label, and positional fallback unavailable (mixed split-page patent)

Rows with `needs_review: true` are **highlighted with a yellow border**.

In [1]:
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")


repo_root : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools


In [2]:
# ── GPU selection + VRAM check ─────────────────────────────────────────────────
# Run this before the cell below if you might be using a GPU for something else
# (another notebook, a training run, etc). It reports free VRAM per card and
# lets you pick which GPU(s) the dual-worker pipeline is allowed to use.
#
# GPU_SELECTION:
#   "auto"    -> use every GPU with enough free VRAM (>= MIN_FREE_GB), one worker each
#   "0"       -> force a single worker on GPU 0 only
#   "1"       -> force a single worker on GPU 1 only
#   "0,1"     -> force both GPUs regardless of current free memory (old default behaviour)
GPU_SELECTION = "1"
MIN_FREE_GB   = 6.0   # YOLO + EasyOCR + 4-bit Qwen2.5-VL-7B need roughly this much headroom

import torch

def _free_gb(idx: int) -> float:
    free_bytes, _total_bytes = torch.cuda.mem_get_info(idx)
    return free_bytes / 1e9

n_gpu = torch.cuda.device_count()
print(f"GPUs visible: {n_gpu}")
free_per_gpu = {}
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    free = _free_gb(i)
    free_per_gpu[i] = free
    flag = "OK" if free >= MIN_FREE_GB else "LOW"
    print(f"  GPU {i} ({name}): {free:.1f} GB free  [{flag}]")

if GPU_SELECTION == "auto":
    selected_gpu_ids = [str(i) for i in range(n_gpu) if free_per_gpu.get(i, 0) >= MIN_FREE_GB]
    if not selected_gpu_ids:
        best = max(free_per_gpu, key=free_per_gpu.get, default=0)
        selected_gpu_ids = [str(best)]
        print(f"\nNo GPU has >= {MIN_FREE_GB} GB free -- falling back to GPU {best} only "
              f"({free_per_gpu.get(best, 0):.1f} GB free). Expect possible OOM/'unavailable' "
              f"qwen_status rows; consider closing other GPU processes first.")
    else:
        print(f"\nAuto-selected GPU(s): {', '.join(selected_gpu_ids)}")
else:
    selected_gpu_ids = [s.strip() for s in GPU_SELECTION.split(",") if s.strip()]
    print(f"\nForced GPU selection: {', '.join(selected_gpu_ids)}")
    for s in selected_gpu_ids:
        idx = int(s)
        free = free_per_gpu.get(idx, 0)
        if free < MIN_FREE_GB:
            print(f"  WARNING: GPU {idx} only has {free:.1f} GB free (< {MIN_FREE_GB} GB) -- "
                  f"this worker may hit OOM. Close other processes on this GPU if possible.")

print(f"\nWorkers to launch: {len(selected_gpu_ids)} -> GPU(s) {selected_gpu_ids}")


GPUs visible: 2
  GPU 0 (NVIDIA GeForce RTX 2080 Ti): 9.1 GB free  [OK]
  GPU 1 (NVIDIA GeForce RTX 2080 Ti): 10.8 GB free  [OK]

Forced GPU selection: 1

Workers to launch: 1 -> GPU(s) ['1']


In [3]:
import os
from src.config_loader import load_config
cfg = load_config()

# Must run BEFORE any sentence_transformers/open_clip/huggingface_hub import:
# huggingface_hub freezes HF_HUB_CACHE as a constant at first import, so
# setting it afterward is silently ignored and weights re-download to the
# default ~/.cache/huggingface instead of the project models/ folder.
os.environ["HF_HUB_CACHE"] = str(cfg["paths"]["siglip_cache"])
os.environ["HF_HOME"]      = str(cfg["paths"]["siglip_cache"])

from src.cross_modal import load_siglip_model, verify_matches

# ── PatentSBERTa — required for T1 dimension classification ──────────────────
# Cached under cfg["paths"]["sbert_cache"] (Patent-Labelling-Tools/models/SBERT)
# so weights stay inside the project instead of ~/.cache/huggingface.
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer("AI-Growth-Lab/PatentSBERTa", cache_folder=str(cfg["paths"]["sbert_cache"]))
print("PatentSBERTa ready.")

# ── SigLIP — required for T2/G1/M1/M2/M3 visual classification ───────────────
USE_SIGLIP = True
siglip_bundle = load_siglip_model(cache_dir=cfg["paths"]["siglip_cache"])
print(f"SigLIP ready on {siglip_bundle[3]}.")


/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


PatentSBERTa ready.
[SigLIP] Loaded ViT-SO400M-14-SigLIP-384 on cuda
SigLIP ready on cuda.


In [ ]:
# ── Batch selector ────────────────────────────────────────────────────────
# Set BATCH_ID to the batch you want to review (see data/batches.xlsx Summary
# sheet — sheets are 1-indexed: Batch_01 .. Batch_05, there is no Batch_00).
BATCH_ID = 2

# Set to a folder name under matched/ to bypass batches.xlsx entirely and test
# against an ad-hoc folder (e.g. a tester batch that has no batches.xlsx sheet).
# Leave as None for a real run driven by BATCH_ID.
OVERRIDE_BATCH_DIR = None   # e.g. "Batch_01_review_tester"

import pandas as pd
from pathlib import Path

cfg = __import__("src.config_loader", fromlist=["load_config"]).load_config()

if OVERRIDE_BATCH_DIR:
    sheet_name = OVERRIDE_BATCH_DIR
    matched_probe = Path(cfg["paths"]["matched"]) / sheet_name
    patent_ids = sorted(p.name for p in matched_probe.iterdir() if p.is_dir())
    batch_df = pd.DataFrame({"patent_id": patent_ids})
    print(f"OVERRIDE: {len(patent_ids)} patents loaded from matched/{sheet_name}/ (no batches.xlsx sheet used)")
else:
    batches_path = Path(cfg["paths"]["data"]) / "batches.xlsx"
    if not batches_path.exists():
        raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

    sheet_name  = f"Batch_{BATCH_ID:02d}"
    batch_df    = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)
    patent_ids  = batch_df["patent_id"].dropna().str.strip().tolist()

    print(f"Batch {BATCH_ID}: {len(patent_ids)} patents loaded from {sheet_name}")
    print(batch_df[["patent_id","company_canonical","prototype_label"]].head(10).to_string(index=False))


In [5]:
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src.reviewer import process_patent

cfg = load_config()
# matched/ is now nested per batch (Stage 00b2 writes to matched/Batch_NN/) —
# use the same sheet_name computed in the batch-selector cell above so this
# notebook looks in the same place 00b2 wrote to.
from pathlib import Path as _Path
matched_dir = _Path(cfg['paths']['matched']) / sheet_name
print(f"matched : {matched_dir}")


matched : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/matched/Batch_01_review_tester


In [6]:
# Excel used for T1 metadata only (title, assignee, years, citations).
# Drawing descriptions are read from data/descriptions.csv by process_patent().
excel_index = load_patseer_excel(cfg['paths']['patseer_excel'])
print(f"Indexed {len(excel_index)} patents.")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer Excel: 1639__dataset_08_06_26.xlsx  (1639 rows, 102 columns)
Columns:
  [  0] 'Record Number'
  [  1] 'Title'
  [  2] 'Abstract'
  [  3] 'Description of Drawings'
  [  4] 'CPC'
  [  5] 'PDF Link'
  [  6] 'Record Type'
  [  7] 'Publication/Issue Date'
  [  8] 'Filing/Application Date'
  [  9] 'Estimated Expiry Date'
  [ 10] 'EFAM Earliest Priority Date'
  [ 11] 'EFAM Earliest Publication Date'
  [ 12] 'Priority Details'
  [ 13] 'Priority Dates (All)'
  [ 14] 'Application No.'
  [ 15] 'Priority Country Code'
  [ 16] 'Priority Year'
  [ 17] 'Register Legal Status'
  [ 18] 'Register Legal Status Date'
  [ 19] 'Summary of Invention'
  [ 20] 'Designated States'
  [ 21] 'Active in Designated States'
  [ 22] 'Field of search'
  [ 23] 'Industry'
  [ 24] 'Tech Domain'
  [ 25] 'Tech Sub Domain'
  [ 26] 'Description'
  [ 27] 'Claims'
  [ 28] 'Number Of Claims'
  [ 29] 'No. of Independent Claims'
  [ 30] 'Independent Claims'
  [ 31] 'First Claim'
  [ 32] 'Advantages of Invention'
  [ 33] 'N

In [7]:
# ── Single-patent diagnostic (optional) ─────────────────────────────────────
# Set INSPECT_ID to the bare patent_id (matches excel_index / descriptions.csv
# keys) — NOT the matched/ folder name, which carries a "_{record_number}"
# suffix. process_patent() resolves the folder internally via
# resolve_patent_image_dir(), and returns the assembled record dict directly
# (no JSON file is written here).
# Leave as None to skip.

INSPECT_ID = "EP3770063A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    data = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    print(f"Inspected: {INSPECT_ID}")
    print(f"Figures: {len(data.get('T3_images', []))}")
    print(f"has_splits: {data.get('has_splits')}")


/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/cross_modal.py:253: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/src/cross_modal.py:268: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Inspected: EP3770063A1
Figures: 7
has_splits: False


In [ ]:
# ── Batch run configuration ──────────────────────────────────────────────────
LIMIT = 30   # int or None — None processes all patents in matched/
# SigLIP and SBERT are always on — see model-loading cell above.
# ─────────────────────────────────────────────────────────────────────────────


In [9]:
# ── Skip flagged crops from SigLIP processing (Cell 5c re-crop flag) ───────
# Cell 5c of 00b2 lets a reviewer flag a crop "re_crop" via the [✂ Re-crop]
# button — those crops, and any already "rejected", must be excluded here
# until a human manually re-crops and clears the flag in crops_mapping.csv.
import functools
import pandas as pd
from src import reviewer as _reviewer

SKIP_QUALITIES = {"rejected", "re_crop"}

# crops_mapping.csv CAN be per-batch: data/<Batch_NN>/crops_mapping_<Batch_NN>.csv
# (named after sheet_name, written by 00b2) — but matched/ and data/ don't always
# get nested in lockstep, so fall back to the flat data/crops_mapping.csv when
# the per-batch copy isn't actually on disk, instead of silently excluding nothing.
_batch_crops_csv = Path(cfg["paths"]["data_matched"]) / sheet_name / f"crops_mapping_{sheet_name}.csv"
_flat_crops_csv  = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
_crops_csv = _batch_crops_csv if _batch_crops_csv.exists() else _flat_crops_csv

if _crops_csv.exists():
    _crops_df = pd.read_csv(_crops_csv)
    skip_files = set(
        _crops_df.loc[_crops_df["crop_quality"].isin(SKIP_QUALITIES), "output"].dropna()
    )
else:
    skip_files = set()

print(f"Reading crop quality from: {_crops_csv}")
print(f"Excluding {len(skip_files)} crop(s) from SigLIP processing (qualities: {sorted(SKIP_QUALITIES)}).")

_reviewer.process_patent = functools.partial(_reviewer.process_patent, skip_files=skip_files)


Reading crop quality from: /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/matched/Batch_01_review_tester/crops_mapping_Batch_01_review_tester.csv
Excluding 0 crop(s) from SigLIP processing (qualities: ['re_crop', 'rejected']).


In [ ]:
from src.reviewer import run_stage01

_run_patent_ids = patent_ids[:LIMIT] if LIMIT else patent_ids
print(f"Processing {len(_run_patent_ids)}/{len(patent_ids)} patent(s) from {sheet_name} (LIMIT={LIMIT}).")

summary_df = run_stage01(
    cfg,
    sbert_model   = sbert,           # loaded in model-loading cell above
    siglip_bundle = siglip_bundle,   # None if SigLIP not loaded
    skip_siglip   = not USE_SIGLIP,
    patent_ids    = _run_patent_ids,   # process only this batch (sliced by LIMIT)
    matched_dir   = matched_dir,  # matched/Batch_NN/ — matches 00b2's per-batch output
)

In [11]:
print(summary_df.to_string(index=False))

              patent_id  match_score  matched  semantic  positional  unmatched  human_required  has_splits  review_required  description_found  t2_labeled  total_crops error
   CA3096221A1_81206876          0.0        0         0           0         14               0       False             True              False          14           14  None
   EP3770063A1_67851083          0.0        0         0           0          7               0       False             True              False           7            7  None
  US12600459B1_99440105          0.0        0         0           0         12               0       False             True              False          12           12  None
US2006198732A1_33154123          0.0        0         0           0          4               0       False             True              False           4            4  None
US2014360830A1_52009426          0.0        0         0           0          9               0       False             True       

In [12]:
# ── Flag crops still needing attention (live, from crops_mapping.csv) ──────
# Purely informational: still process every crop normally below, this just
# surfaces which ones need a human look. Reads the SAME live crops_mapping.csv
# (needs_review + crop_quality columns) that src/reviewer.py's review_flags
# mechanism now bakes into each T3_images entry and into source_patents.xlsx —
# so this print always agrees with what ends up in the spreadsheet.
# (Not needs_human_review.csv: that's a one-time pre-review snapshot that goes
# stale immediately and never gains crop_quality at all — see reviewer.py's
# _load_review_flags() docstring.)
import pandas as pd

# Same flat-vs-per-batch fallback as the SKIP_QUALITIES cell above, so this
# print always agrees with what review_flags actually used during the run.
_batch_crops_csv = Path(cfg["paths"]["data_matched"]) / sheet_name / f"crops_mapping_{sheet_name}.csv"
_flat_crops_csv  = Path(cfg["paths"]["data"]) / "crops_mapping.csv"
_crops_csv = _batch_crops_csv if _batch_crops_csv.exists() else _flat_crops_csv

if _crops_csv.exists():
    _crops_df = pd.read_csv(_crops_csv)
    if "crop_quality" not in _crops_df.columns:
        _crops_df["crop_quality"] = ""
    _crops_df["crop_quality"] = _crops_df["crop_quality"].fillna("")
    _needs_attention = (_crops_df["needs_review"] == True) | (_crops_df["crop_quality"] != "")
    _flagged_df = _crops_df.loc[_needs_attention]
else:
    _flagged_df = pd.DataFrame(columns=["patent_id", "output"])

print(f"{len(_flagged_df)} crop(s) in this batch still need attention "
      f"(from {_crops_csv}).")

if not _flagged_df.empty and not summary_df.empty:
    per_patent_flagged = (
        _flagged_df.groupby("patent_id")["output"]
        .apply(list)
        .rename("crops_needing_attention")
    )
    flagged_summary = summary_df.set_index("patent_id").join(per_patent_flagged, how="left")
    flagged_summary["crops_needing_attention"] = flagged_summary["crops_needing_attention"].apply(
        lambda v: v if isinstance(v, list) else []
    )
    flagged_summary["n_needing_attention"] = flagged_summary["crops_needing_attention"].apply(len)
    display(flagged_summary[flagged_summary["n_needing_attention"] > 0]
            [["n_needing_attention", "crops_needing_attention"]])


128 crop(s) in this batch still need attention (from /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/matched/Batch_01_review_tester/crops_mapping_Batch_01_review_tester.csv).


,n_needing_attention,crops_needing_attention
patent_id,,


In [13]:
# ── Single-patent diagnostic — per-figure match table ───────────────────────
# Set INSPECT_ID to a bare patent_id from this batch to see its T3_images
# table. process_patent() returns its record in-memory only —
# source_patents.xlsx is the persisted output. This calls process_patent()
# live, same as the diagnostic cell near the top of the notebook, just
# with a fuller per-figure table.

INSPECT_ID = None   # e.g. "EP3770063A1"

if INSPECT_ID:
    from src.reviewer import process_patent
    d    = process_patent(
        INSPECT_ID, cfg, excel_index, matched_dir,
        sbert_model=sbert, siglip_bundle=siglip_bundle, skip_siglip=not USE_SIGLIP,
    )
    figs = d.get("T3_images", [])
    icons = {
        "matched":        "🟢",
        "semantic":       "🔵",
        "positional":     "🟡",
        "unmatched":      "🔴",
        "human_required": "⚫",
    }
    print(f"\n{INSPECT_ID}  |  splits={d.get('has_splits')}")
    print(f"{'FILE':<32} {'STATUS':<16} {'CONF':>5}  {'T2_PER':<20}  DESC[:50]")
    print("-" * 110)
    for f in figs:
        icon = icons.get(f.get("match_status", ""), "❓")
        t2   = f.get("T2_predictions") or {}
        per  = (t2.get("per") or {}).get("value", "—")
        desc = (f.get("matched_description") or "")[:50]
        conf = f.get("composite_confidence") or f.get("match_confidence") or 0.0
        print(
            f"{f['file']:<32} {icon} {f.get('match_status',''):<14} "
            f"{float(conf):>5.2f}  {per:<20}  {desc}"
        )


---
## Finalize this batch

**Restored** — reconstructed from this notebook's earlier version (only partial source was on hand, so this is a clean rewrite of the same intent, not a byte-identical recovery). Run after you've set `human_review_status` to `"approved"` for the patents you want to keep, directly in `batches.xlsx`.

In [14]:
# ── Copy approved files → reviewed/{company}/{prototype}/{patent_id}/ ──────
# Run this after you have finished human review for this batch.
# Only patents with human_review_status == "approved" in batches.xlsx (this
# batch's sheet) are copied. Raw files are never touched — source is
# matched/<sheet_name>/{patent_id}/ (00b2's per-batch nested output), which
# matched_dir already points at (set in the batch-selector cells above).
import shutil
from pathlib import Path
import pandas as pd

batches_path  = Path(cfg["paths"]["data"]) / "batches.xlsx"
batch_df_live = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str)

approved = batch_df_live[
    batch_df_live["human_review_status"].str.strip().str.lower() == "approved"
]
print(f"{len(approved)}/{len(batch_df_live)} patents in {sheet_name} marked approved.")

reviewed_root = Path(cfg["paths"]["reviewed"])
copied, missing = 0, 0
for _, row in approved.iterrows():
    pid       = str(row["patent_id"]).strip()
    company   = str(row["company_canonical"]).strip()
    prototype = str(row["prototype_label"]).strip()
    src_dir   = matched_dir / pid
    dst_dir   = reviewed_root / company / prototype / pid

    if not src_dir.exists():
        print(f"  ⚠ missing matched/ folder for {pid} — skipped")
        missing += 1
        continue
    dst_dir.parent.mkdir(parents=True, exist_ok=True)
    if dst_dir.exists():
        shutil.rmtree(dst_dir)
    shutil.copytree(src_dir, dst_dir)
    copied += 1

print(f"\nCopied: {copied}   Missing source: {missing}")


ValueError: Worksheet named 'Batch_01_review_tester' not found

In [ ]:
# ── Collect batch outputs → <sheet_name_lower>/ ────────────────────────────
# Copies this batch's label JSONs + any review HTML exports into a flat
# top-level folder for your own validation/inspection. Purely a convenience
# snapshot — skip once you trust the pipeline and are running all batches
# end-to-end without spot-checking each one individually.
import shutil
from pathlib import Path

BATCH_OUT = Path(cfg["paths"]["base"]) / sheet_name.lower()
BATCH_OUT.mkdir(parents=True, exist_ok=True)

# 1. label JSONs for this batch (one per patent, written by reviewer.py)
labels_src = Path(cfg["paths"]["labels"])
labels_dst = BATCH_OUT / "labels"
labels_dst.mkdir(parents=True, exist_ok=True)
n_labels = 0
for pid in patent_ids:
    f = labels_src / f"{pid}.json"
    if f.exists():
        shutil.copy2(f, labels_dst / f.name)
        n_labels += 1

# 2. review HTML exports for this batch, if any were generated
html_src = Path(cfg["paths"]["data"]) / "review_html" / sheet_name
html_dst = BATCH_OUT / "review_html"
n_html = 0
if html_src.exists():
    shutil.copytree(html_src, html_dst, dirs_exist_ok=True)
    n_html = len(list(html_dst.glob("*")))

print(f"Collected {n_labels} label JSON(s) and {n_html} HTML file(s) → {BATCH_OUT}")
